# Import Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
# df_debit_clean = pd.read_parquet("data/base/df_debit_clean_v2.parquet")
# df_freq = pd.read_parquet("data/feature_engineering/debit/v2/df_freq.parquet")
# df_monetary = pd.read_parquet("data/feature_engineering/debit/v2/df_monetary.parquet")
# df_time_diff = pd.read_parquet("data/feature_engineering/debit/v2/df_time_diff.parquet")
# df_unique_count = pd.read_parquet("data/feature_engineering/debit/v2/df_unique_count.parquet")

# df_debit_clean = pd.read_parquet("data/base/df_debit_clean_all_sample_v1.parquet")
# df_freq = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_freq.parquet")
# df_monetary = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_monetary.parquet")
# df_time_diff = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_time_diff.parquet")
# df_unique_count = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_unique_count.parquet")

df_debit_clean = pd.read_parquet("data/base/df_debit_union_20250629.parquet")
df_freq_pos_mode = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_freq_pos_mode.parquet")
df_monetary_pos_mode = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_monetary_pos_mode.parquet")
df_high_risk_label = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_high_risk_label.parquet")
df_duration_since_first_trnx = pd.read_parquet("data/feature_engineering/debit/v2/clean/df_duration_since_first_trnx.parquet")

In [3]:
channel_cols = [
    'Debit_No'
	, 'Transaction Serial No'
    , 'Transaction Datetime'
    , 'Transaction Amount'
    , 'MCC'
    , 'MCC Details'
    , 'MCC Trnx Category Code'
    , 'MCC Category'
    , 'Country Code'
    , 'Card Acceptor Terminal ID'
    , 'Card Acceptor Name'
    , 'Card Acceptor City'
    , 'Card Acceptor Region'
    , 'Card Acceptor Country Code'
    , 'Cat Card Acceptor Name'
    , 'Currency Code'
    , 'Confirmed'
]

# Join Data

In [5]:
from src.debit_card_config import (
    time_shift_config,
    time_windows,
    freq_config,
    monetary_config_1,
    monetary_config_2,
    monetary_config_3,
    monetary_config_4,
    monetary_config_5,
    monetary_config_6,
    unique_count_config
)
from src.debit_card_config import (
    dynamic_high_risk_config,
    freq_config_pos_mode,
    monetary_config_pos_mode,
    duration_since_first_trnx_config,
)
import warnings
warnings.filterwarnings('ignore')

all_monetary_configs = (
    monetary_config_1 + monetary_config_2 + monetary_config_3 +
    monetary_config_4 + monetary_config_5 + monetary_config_6
)

In [6]:
# Define the common keys for merging
merge_keys = ["Transaction Serial No", "Debit_No"]

# Helper function to extract columns based on config and time windows
def extract_columns(config, time_windows):
    result = []
    for cfg in config:
        for win in time_windows:
            if "windows" in cfg and win in cfg["windows"]:
                result.append(cfg["windows"][win])
    return result

# Extract columns
# time_shift_cols = list(time_shift_config.keys())
# freq_cols = extract_columns(freq_config, time_windows)
# unique_count_cols = extract_columns(unique_count_config, time_windows)
# monetary_cols = extract_columns(all_monetary_configs, time_windows)
# monetary_cols = [x for x in monetary_cols if x not in ['Sum_Amt_L15M', 'Sum_Amt_L1H', 'Sum_Amt_L1D', 'Sum_Amt_L7D']]

freq_pos_mode_cols = extract_columns(freq_config_pos_mode, time_windows)
monetary_pos_mode_cols = extract_columns(monetary_config_pos_mode, time_windows)
duration_since_first_trnx_cols = [
    col for col in df_duration_since_first_trnx.columns if "DurationSinceFirstTrnx" in col
]
high_risk_category_col = [
    col for col in df_high_risk_label.columns if f"IsTop{dynamic_high_risk_config["top_n"]}HighRisk" in col
]

In [7]:
from functools import reduce

# Subset dataframes
# df_time_diff = df_time_diff[merge_keys + time_shift_cols]
# df_freq = df_freq[merge_keys + freq_cols]
# df_monetary = df_monetary[merge_keys + monetary_cols]
# df_unique_count = df_unique_count[merge_keys + unique_count_cols]
df_duration_since_first_trnx = df_duration_since_first_trnx[merge_keys + duration_since_first_trnx_cols]
df_high_risk_label = df_high_risk_label[merge_keys + high_risk_category_col]
df_freq_pos_mode = df_freq_pos_mode[merge_keys + freq_pos_mode_cols]
df_monetary_pos_mode = df_monetary_pos_mode[merge_keys + monetary_pos_mode_cols]

# Merge all feature dataframes
# dfs_to_merge = [df_time_diff, df_freq, df_monetary, df_unique_count]
dfs_to_merge = [
    df_duration_since_first_trnx,
    df_high_risk_label,
    df_freq_pos_mode,
    df_monetary_pos_mode
]
df_debit_final = reduce(
    lambda left, right: pd.merge(left, right, on=merge_keys, how="outer"), dfs_to_merge
)

# Additional TSCF Features
tscf_cols = [col for col in df_debit_clean.columns if col not in channel_cols + [
    'CustomerEwaletNumber',
    'CCBrnAccount'
    ]
]
selected_channel_cols = [
    'Debit_No'
	, 'Transaction Serial No'
    , 'Transaction Datetime'
    , 'Transaction Amount'
    # , 'MCC'
    # , 'MCC Details'
    # , 'MCC Trnx Category Code'
    , 'MCC Category'
    , 'Country Code'
    # , 'Card Acceptor City'
    # , 'Card Acceptor Region'
    # , 'Card Acceptor Country Code'
    , 'Cat Card Acceptor Name'
    , 'Currency Code'
    , 'Confirmed'
]

In [8]:
# Merge with additional features
df_debit_final = df_debit_final.merge(
    df_debit_clean[selected_channel_cols + tscf_cols],
    on=merge_keys,
    how="left",
)

In [9]:
len(channel_cols + tscf_cols)

18

In [10]:
df_debit_final['Confirmed'].fillna(0.0, inplace=True)
# df_debit_final.to_parquet("data/feature_engineering/debit/df_debit_final_v2.parquet")
# df_debit_final.to_parquet("data/feature_engineering/debit/df_debit_final_v2__all_clean.parquet")

In [15]:
all_derived_feats = [col for col in df_debit_final.columns if col not in (channel_cols + tscf_cols)]

# Concat/Union Final Data

In [12]:
df_debit_clean_1 = pd.read_parquet("data/base/df_debit_clean_all_sample_v1.parquet")
df_debit_clean_2 = pd.read_parquet("data/base/df_debit_clean_v2.parquet")

In [15]:
df_debit_clean_1 = df_debit_clean_1[['Transaction Serial No','Transaction Datetime']]
df_debit_clean_2 = df_debit_clean_2[['Transaction Serial No','Transaction Datetime']]

In [19]:
df_union_debit_raw = pd.concat([df_debit_clean_1, df_debit_clean_2], axis=0).reset_index(drop=True)

In [16]:
df_debit_mix = pd.read_parquet("data/feature_engineering/debit/df_debit_final_v2.parquet")
df_debit_good = pd.read_parquet("data/feature_engineering/debit/df_debit_final_v2__all_clean.parquet")

In [17]:
print(df_debit_mix.shape)
print(df_debit_good.shape)

(380099, 264)
(334635, 262)


In [18]:
set(df_debit_mix.columns) - set(df_debit_good.columns)

{'Card Acceptor Name', 'Card Acceptor Terminal ID'}

In [20]:
df_debit_mix.drop(
    columns=['Card Acceptor Name', 'Card Acceptor Terminal ID'],
    axis=1,
    inplace=True
)

In [21]:
print(df_debit_mix.shape)
print(df_debit_good.shape)

(380099, 262)
(334635, 262)


In [22]:
df_union_debit = pd.concat([df_debit_mix, df_debit_good], axis=0).reset_index(drop=True)

In [24]:
df_union_debit_final = df_union_debit.merge(
    df_union_debit_raw,
    on='Transaction Serial No',
    how='left'
)

# Create Additional Feature: TrnxHour Rolling

In [28]:
from tqdm import tqdm

def compute_grouped_rolling_avg_trnx_hour(
    df: pd.DataFrame,
    group_col: str,
    datetime_col: str,
    windows: list[str]
) -> pd.DataFrame:
    """
    Compute grouped rolling average of transaction hour per group
    over time windows
    """
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df['TrnxHour'] = df[datetime_col].dt.hour

    # container for results
    result = []

    # apply group-wise rolling
    for group_value, group_df in tqdm(
        df.groupby(group_col), desc="Processing TrnxHour Rolling Avg"):
        group_df = group_df.sort_values(datetime_col).set_index(datetime_col)
        
        # for each window, apply group-wise rolling
        for window in windows:
            col_name = f'AvgTrnxHourL{window}'
            group_df[col_name] = group_df['TrnxHour'].rolling(
                window=window, min_periods=1
            ).mean()

        # restore group column
        group_df[group_col] = group_value
        result.append(group_df.reset_index())

    # concatenate all groupes
    df_final = pd.concat(result, ignore_index=False)
    return df_final

In [29]:
windows=['15min','1h','1d','14d','30d']

df_union_debit_final = compute_grouped_rolling_avg_trnx_hour(
    df=df_union_debit_final,
    group_col='Debit_No',
    datetime_col='Transaction Datetime',
    windows=windows
)

Processing TrnxHour Rolling Avg: 100%|██████████| 26027/26027 [02:18<00:00, 187.44it/s]


In [30]:
print(df_union_debit_final.shape)

(714734, 269)


In [33]:
[col for col in df_union_debit_final.columns if any(w in col for w in windows)]

['AvgTrnxHourL15min',
 'AvgTrnxHourL1h',
 'AvgTrnxHourL1d',
 'AvgTrnxHourL14d',
 'AvgTrnxHourL30d']

# Save Final Data

In [11]:
# df_union_debit_final.to_parquet("data/feature_engineering/debit/v2/df_union_debit_final_20250620.parquet")
df_union_debit = pd.read_parquet("data/feature_engineering/debit/v2/df_union_debit_final_20250620.parquet")

In [12]:
df_union_debit.Confirmed.value_counts(normalize=True)

Confirmed
0.0    0.999317
1.0    0.000683
Name: proportion, dtype: float64

In [13]:
len(df_union_debit)

714734

In [20]:
# Merge with additional features
df_debit_final_final = df_union_debit.merge(
    df_debit_final[merge_keys+all_derived_feats],
    on=merge_keys,
    how="left",
)

In [23]:
df_debit_final_final.to_parquet("data/feature_engineering/debit/v2/df_union_debit_final_20250629.parquet")